# Python Design Patterns
Covers: Singleton, Factory, Builder, Decorator, Adapter, Facade, Observer, Strategy, Command, Iterator, Context Manager, Descriptor

## 1. Creational — Singleton, Factory, Builder

In [ ]:
# Singleton
class SingletonMeta(type):
    _instances = {}
    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            cls._instances[cls] = super().__call__(*args, **kwargs)
        return cls._instances[cls]

class AppConfig(metaclass=SingletonMeta):
    def __init__(self):
        self.settings = {'debug': False, 'version': '1.0'}
    def get(self, key, default=None):
        return self.settings.get(key, default)
    def set(self, key, value):
        self.settings[key] = value

c1 = AppConfig()
c2 = AppConfig()
c1.set('debug', True)
print('Singleton: c1 is c2:', c1 is c2)
print('c2.get(debug):', c2.get('debug'))  # True — same instance

# Factory
class Shape:
    _registry = {}
    
    def __init_subclass__(cls, shape_type=None, **kwargs):
        super().__init_subclass__(**kwargs)
        if shape_type:
            Shape._registry[shape_type] = cls
    
    @classmethod
    def create(cls, shape_type, **kwargs):
        if shape_type not in cls._registry:
            raise ValueError(f'Unknown shape: {shape_type}')
        return cls._registry[shape_type](**kwargs)
    
    def area(self): raise NotImplementedError

class Circle(Shape, shape_type='circle'):
    def __init__(self, radius): self.radius = radius
    def area(self): import math; return math.pi * self.radius ** 2
    def __repr__(self): return f'Circle(r={self.radius})'

class Rectangle(Shape, shape_type='rectangle'):
    def __init__(self, width, height): self.width = width; self.height = height
    def area(self): return self.width * self.height
    def __repr__(self): return f'Rectangle({self.width}x{self.height})'

print('\nFactory:')
for spec in [('circle', {'radius': 5}), ('rectangle', {'width': 4, 'height': 6})]:
    shape = Shape.create(spec[0], **spec[1])
    print(f'  {shape}: area={shape.area():.2f}')

In [ ]:
# Builder — fluent interface
class QueryBuilder:
    def __init__(self):
        self._table = ''
        self._columns = []
        self._conditions = []
        self._order_by = None
        self._limit = None
        self._offset = None
    
    def select(self, *columns):
        self._columns = list(columns)
        return self
    
    def from_table(self, table):
        self._table = table
        return self
    
    def where(self, condition):
        self._conditions.append(condition)
        return self
    
    def order_by(self, column, direction='ASC'):
        self._order_by = f'{column} {direction}'
        return self
    
    def limit(self, n):
        self._limit = n
        return self
    
    def offset(self, n):
        self._offset = n
        return self
    
    def build(self):
        cols = ', '.join(self._columns) if self._columns else '*'
        q = f'SELECT {cols} FROM {self._table}'
        if self._conditions:
            q += ' WHERE ' + ' AND '.join(self._conditions)
        if self._order_by:
            q += f' ORDER BY {self._order_by}'
        if self._limit:
            q += f' LIMIT {self._limit}'
        if self._offset:
            q += f' OFFSET {self._offset}'
        return q

query = (QueryBuilder()
    .select('id', 'name', 'email')
    .from_table('users')
    .where('age > 18')
    .where('active = true')
    .order_by('name')
    .limit(10)
    .offset(20)
    .build())

print('Builder query:')
print(query)

## 2. Structural — Decorator, Adapter, Facade

In [ ]:
import functools
import time

# Decorator pattern (function decorator)
def retry(max_attempts=3, delay=0.05, exceptions=(Exception,)):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for attempt in range(max_attempts):
                try:
                    return func(*args, **kwargs)
                except exceptions as e:
                    if attempt == max_attempts - 1:
                        raise
                    print(f'  Attempt {attempt+1} failed: {e}. Retrying...')
                    time.sleep(delay)
        return wrapper
    return decorator

def cache(func):
    memo = {}
    @functools.wraps(func)
    def wrapper(*args):
        if args not in memo:
            memo[args] = func(*args)
        return memo[args]
    wrapper.cache = memo
    return wrapper

import random
call_count = 0

@retry(max_attempts=5, delay=0.01, exceptions=(ValueError,))
def flaky_function():
    global call_count
    call_count += 1
    if random.random() < 0.6:  # 60% failure rate
        raise ValueError('Random failure')
    return 'Success!'

@cache
def expensive_fib(n):
    if n < 2: return n
    return expensive_fib(n-1) + expensive_fib(n-2)

print('Retry decorator:')
try:
    result = flaky_function()
    print(f'Result: {result} (took {call_count} attempts)')
except ValueError:
    print(f'Failed after {call_count} attempts')

print('\nCache decorator:')
print('fib(30):', expensive_fib(30))
print('Cache size:', len(expensive_fib.cache))

# Adapter
class LegacyLogger:
    def log_message(self, level, msg, timestamp):
        print(f'[{timestamp}] {level}: {msg}')

class ModernLogger:
    def info(self, msg): raise NotImplementedError
    def error(self, msg): raise NotImplementedError

class LoggerAdapter(ModernLogger):
    def __init__(self, legacy):
        self._legacy = legacy
    def info(self, msg):
        from datetime import datetime
        self._legacy.log_message('INFO', msg, datetime.now().strftime('%H:%M:%S'))
    def error(self, msg):
        from datetime import datetime
        self._legacy.log_message('ERROR', msg, datetime.now().strftime('%H:%M:%S'))

print('\nAdapter:')
logger = LoggerAdapter(LegacyLogger())
logger.info('Application started')
logger.error('Something went wrong')

## 3. Behavioral — Observer, Strategy, Command

In [ ]:
from typing import Any, Callable

# Observer
class EventEmitter:
    def __init__(self):
        self._listeners: dict[str, list[Callable]] = {}
    
    def on(self, event: str, callback: Callable) -> None:
        self._listeners.setdefault(event, []).append(callback)
    
    def off(self, event: str, callback: Callable) -> None:
        if event in self._listeners:
            self._listeners[event].remove(callback)
    
    def emit(self, event: str, data: Any = None) -> None:
        for cb in self._listeners.get(event, []):
            cb(data)

class OrderService(EventEmitter):
    def place_order(self, item, qty):
        order = {'item': item, 'qty': qty, 'id': id(item)}
        self.emit('order.placed', order)
        return order
    
    def cancel_order(self, order_id):
        self.emit('order.cancelled', {'id': order_id})

service = OrderService()
service.on('order.placed', lambda d: print(f'  [EMAIL] Order placed: {d["item"]} x{d["qty"]}'))
service.on('order.placed', lambda d: print(f'  [INVENTORY] Reserve {d["qty"]} units of {d["item"]}'))
service.on('order.cancelled', lambda d: print(f'  [REFUND] Processing refund for order {d["id"]}'))

print('Observer pattern:')
order = service.place_order('Widget', 3)
service.cancel_order(order['id'])

# Strategy
class Sorter:
    def __init__(self, strategy=None):
        self.strategy = strategy or sorted
    
    def sort(self, data):
        return self.strategy(data)

def reverse_sort(data): return sorted(data, reverse=True)
def length_sort(data): return sorted(data, key=len)

print('\nStrategy pattern:')
words = ['banana', 'apple', 'cherry', 'date', 'elderberry']
sorter = Sorter()
print('Default:', sorter.sort(words))
sorter.strategy = reverse_sort
print('Reverse:', sorter.sort(words))
sorter.strategy = length_sort
print('By length:', sorter.sort(words))

# Command with undo
class TextEditor:
    def __init__(self): self.text = ''
    def insert(self, text, pos): self.text = self.text[:pos] + text + self.text[pos:]
    def delete(self, pos, n):
        deleted = self.text[pos:pos+n]
        self.text = self.text[:pos] + self.text[pos+n:]
        return deleted

class InsertCmd:
    def __init__(self, editor, text, pos):
        self.editor = editor; self.text = text; self.pos = pos
    def execute(self): self.editor.insert(self.text, self.pos)
    def undo(self): self.editor.delete(self.pos, len(self.text))

editor = TextEditor()
history = []

def execute(cmd):
    cmd.execute()
    history.append(cmd)

def undo():
    if history:
        history.pop().undo()

print('\nCommand pattern (undo):')
execute(InsertCmd(editor, 'Hello', 0))
execute(InsertCmd(editor, ' World', 5))
print('After inserts:', editor.text)
undo()
print('After undo:', editor.text)
undo()
print('After undo:', editor.text)